# Delta Lake Practice

Hands-on tour of Delta Lake: creating a table, appends, schema evolution, the transaction log (DESCRIBE HISTORY), time travel, MERGE, and restoring an old version.

In [0]:
from pyspark.sql import functions as F

# Where does this notebook write? Check the current catalog/schema first
print(spark.sql("SELECT current_catalog() AS catalog, current_schema() AS default_schema").collect())

# A dedicated schema for this practice notebook
spark.sql("CREATE SCHEMA IF NOT EXISTS delta_practice")
TABLE = "delta_practice.employees"
print(f"Tables will be created as {TABLE}")

[Row(catalog='workspace', default_schema='default')]
Tables will be created as delta_practice.employees


In [0]:
employees = spark.createDataFrame(
    [
        (101, "Ravi",  "IT",    80000),
        (102, "Priya", "IT",    95000),
        (104, "Sita",  "HR",    60000),
        (105, "Arjun", "HR",    75000),
        (107, "Kiran", "Sales", 70000),
        (108, "Meena", "Sales", 90000),
    ],
    schema="emp_id LONG, name STRING, department STRING, salary LONG",
)

# Write it as a Delta table (saveAsTable creates a managed Delta table in Unity Catalog)
employees.write.format("delta").mode("overwrite").saveAsTable(TABLE)

display(spark.table(TABLE))

emp_id,name,department,salary
101,Ravi,IT,80000
102,Priya,IT,95000
104,Sita,HR,60000
105,Arjun,HR,75000
107,Kiran,Sales,70000
108,Meena,Sales,90000


In [0]:
# Every write to a Delta table is recorded as a new "version" in the transaction log
display(spark.sql(f"DESCRIBE HISTORY {TABLE}"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-11-03T04:30:29.000Z,78487087375229,garidiyaswanth1697@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""b3215255-9ba7-41be-a11d-3ca834353635"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-4dd85e21-c8c2-4576-8f8d-8c91ab66f069"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-c0707a2b-1b82-419c-9f87-62a4188f6bc8""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(3522870837325771),baadfb8b-0ffe-4521-bd45-5a91ebcb6c89,1103-042331-qg8zfdq9-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 6, numOutputBytes -> 1617)",null,Databricks-Runtime/19.12.x-aarch64-photon-scala2.13


In [0]:
new_hires = spark.createDataFrame(
    [
        (110, "Anu",   "IT",    60000),
        (109, "Rahul", "Sales", 65000),
    ],
    schema="emp_id LONG, name STRING, department STRING, salary LONG",
)

# Append keeps existing data and commits a new version
new_hires.write.format("delta").mode("append").saveAsTable(TABLE)

# Schema evolution: add a brand-new column, allowed with overwriteSchema
tmp = spark.table(TABLE).withColumn("bonus", (F.col("salary") * 0.10).cast("long"))
tmp.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE)

display(spark.table(TABLE))

emp_id,name,department,salary,bonus
101,Ravi,IT,80000,8000
102,Priya,IT,95000,9500
104,Sita,HR,60000,6000
105,Arjun,HR,75000,7500
107,Kiran,Sales,70000,7000
108,Meena,Sales,90000,9000
110,Anu,IT,60000,6000
109,Rahul,Sales,65000,6500


In [0]:
# Time travel: read the table as it was at version 0 (right after creation)
print("Version 0 (original 6 rows, no bonus column):")
display(spark.read.option("versionAsOf", 0).table(TABLE))

print("Latest version (8 rows + bonus column):")
display(spark.table(TABLE))

Version 0 (original 6 rows, no bonus column):


emp_id,name,department,salary
101,Ravi,IT,80000
102,Priya,IT,95000
104,Sita,HR,60000
105,Arjun,HR,75000
107,Kiran,Sales,70000
108,Meena,Sales,90000


Latest version (8 rows + bonus column):


emp_id,name,department,salary,bonus
101,Ravi,IT,80000,8000
102,Priya,IT,95000,9500
104,Sita,HR,60000,6000
105,Arjun,HR,75000,7500
107,Kiran,Sales,70000,7000
108,Meena,Sales,90000,9000
110,Anu,IT,60000,6000
109,Rahul,Sales,65000,6500


In [0]:
updates = spark.createDataFrame(
    [
        (101, "Ravi", "IT", 85000),   # existing employee -> UPDATE salary
        (200, "Zara", "IT", 88000),   # brand-new employee -> INSERT
    ],
    schema="emp_id LONG, name STRING, department STRING, salary LONG",
)
updates.createOrReplaceTempView("updates")

spark.sql(f"""
    MERGE INTO {TABLE} AS t
    USING updates AS s
    ON t.emp_id = s.emp_id
    WHEN MATCHED THEN UPDATE SET t.salary = s.salary
    WHEN NOT MATCHED THEN
      INSERT (emp_id, name, department, salary)
      VALUES (s.emp_id, s.name, s.department, s.salary)
""")

display(spark.table(TABLE))

emp_id,name,department,salary,bonus
102,Priya,IT,95000,9500
104,Sita,HR,60000,6000
105,Arjun,HR,75000,7500
107,Kiran,Sales,70000,7000
108,Meena,Sales,90000,9000
110,Anu,IT,60000,6000
109,Rahul,Sales,65000,6500
101,Ravi,IT,85000,8000
200,Zara,IT,88000,null


In [0]:
# Table metadata: location, numFiles, sizeInBytes, current version, min/max reader/writer versions
display(spark.sql(f"DESCRIBE DETAIL {TABLE}"))

# Time travel can also RESTORE: roll the table back to version 1
# (the state right after the append, before the bonus column and the MERGE).
# Uncomment the two lines below and re-run this cell to try it —
# it discards everything committed after version 1 (here: bonus column, Zara, Ravi's raise).
# spark.sql(f"RESTORE TABLE {TABLE} TO VERSION 1")

# display(spark.table(TABLE))

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,117dd624-1a81-4a84-a467-0549708029f8,workspace.delta_practice.employees,null,,2026-11-03T04:30:24.264Z,2026-11-03T04:31:55.000Z,List(),List(),1,2717,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> b3215255-9ba7-41be-a11d-3ca834353635, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-4dd85e21-c8c2-4576-8f8d-8c91ab66f069, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-c0707a2b-1b82-419c-9f87-62a4188f6bc8)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",true
